# MINDTRACE — MentalBERT 7-Class Model Full-Scale Reproduction

**Authoritative Source**: `notebooks/text_model.ipynb`  
**Target Hardware**: Google Colab with **Tesla T4 GPU** (or V100/A100)  
**Expected Duration**: ~50 minutes on Tesla T4  

### Historical Recorded Benchmark Targets:
- **Test Accuracy**: **84.26%**
- **Macro F1**: **82.90%**
- **Weighted F1**: **84.21%**
- **Suicidal-Class Recall**: **71.0%** (Precision: 75%, F1: 73%)

---

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers torch scikit-learn pandas numpy datasets matplotlib seaborn

In [ ]:
# CELL 2 — Check GPU Environment
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Compute Device:', device)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f'GPU: {gpu_name} ({vram_gb} GB VRAM)')
else:
    print('WARNING: Running on CPU! To enable GPU: Runtime > Change runtime type > T4 GPU.')

In [ ]:
# CELL 3 — Hugging Face Hub Authentication (Non-blocking)
import os
from huggingface_hub import login

# Try Colab secrets first, then environment, then fallback research token
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or userdata.get('Hugging_face')
except Exception:
    pass

if not hf_token:
    hf_token = os.getenv('HF_TOKEN', 'REMOVED_HF_TOKEN_SEE_ENV')

try:
    login(token=hf_token)
    print('Authenticated with Hugging Face Hub successfully.')
except Exception as e:
    print(f'Hugging Face login note: {e}')

In [ ]:
# CELL 4 — Load Full Dataset (52,680 records)
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print('Loading Sentiment Analysis for Mental Health dataset...')
if os.path.exists('sentiment_mental_health.csv'):
    df_raw = pd.read_csv('sentiment_mental_health.csv', encoding='latin-1')
    print('Loaded from local CSV file.')
else:
    ds = load_dataset('AhmedSSoliman/sentiment-analysis-for-mental-health-Combined-Data')['train']
    df_raw = ds.to_pandas()
    print('Loaded from Hugging Face Hub dataset mirror.')

print(f'Raw dataset records: {len(df_raw)}')

text_col = 'statement' if 'statement' in df_raw.columns else 'text'
label_col = 'status' if 'status' in df_raw.columns else 'label'

df_clean = pd.DataFrame({
    'text': df_raw[text_col],
    'label': df_raw[label_col]
}).dropna()

print(f'Clean dataset records (after dropna): {len(df_clean)}')
assert len(df_clean) >= 52680, f'Expected >= 52,680 records, got {len(df_clean)}'

# Encode labels alphabetically (exact original mapping)
le = LabelEncoder()
df_clean['label_encoded'] = le.fit_transform(df_clean['label'])
class_names = list(le.classes_)

print('
Target Classes (Exact 7 Classes):')
for idx, name in enumerate(class_names):
    count = (df_clean['label'] == name).sum()
    print(f'  {idx} -> {name:20s} ({count:,} records)')

# Exact string preprocessing
X = [str(x).strip() if x is not None and str(x).strip() else 'empty' for x in df_clean['text']]
y = df_clean['label_encoded'].tolist()

# Stratified 80 / 10 / 10 split (exact seed 42)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'
Splits Verified:')
print(f'  Train      : {len(X_train):,} samples (Expected: 42,144)')
print(f'  Validation : {len(X_val):,} samples (Expected: 5,268)')
print(f'  Held-out Test: {len(X_test):,} samples (Expected: 5,268)')
assert len(X_train) == 42144
assert len(X_val) == 5268
assert len(X_test) == 5268

In [ ]:
# CELL 5 — Tokenize with MentalBERT
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'mental/mental-bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)

class MentalHealthDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print('Tokenizing train, validation, and test sets (max_length=128)...')
def tokenize_batch(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')

train_enc = tokenize_batch(X_train)
val_enc   = tokenize_batch(X_val)
test_enc  = tokenize_batch(X_test)

train_loader = DataLoader(MentalHealthDataset(train_enc, y_train), batch_size=16, shuffle=True)
val_loader   = DataLoader(MentalHealthDataset(val_enc, y_val), batch_size=16, shuffle=False)
test_loader  = DataLoader(MentalHealthDataset(test_enc, y_test), batch_size=16, shuffle=False)

print(f'Train batches      : {len(train_loader)} (Expected: 2,634)')
print(f'Validation batches : {len(val_loader)} (Expected: 330)')
print(f'Test batches       : {len(test_loader)} (Expected: 330)')

In [ ]:
# CELL 6 — Full 3-Epoch Training Loop with Best-Model Tracking
import time
import json
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(class_names),
    token=hf_token
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

history = []
best_val_acc = 0.0

print(f'Starting full-scale MentalBERT training for {EPOCHS} epochs on {device}...')
print('=' * 60)

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 200 == 0 or (batch_idx + 1) == len(train_loader):
            elapsed = time.time() - start_time
            eta = (elapsed / (batch_idx + 1)) * (len(train_loader) - batch_idx - 1)
            print(f'  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {loss.item():.4f} | Train Acc: {correct/total:.4f} | ETA: {eta/60:.1f}m')

    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            val_correct += (outputs.logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = correct / total
    val_acc = val_correct / val_total
    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    epoch_time = (time.time() - start_time) / 60

    print('
' + '=' * 60)
    print(f'Epoch {epoch+1}/{EPOCHS} Complete ({epoch_time:.1f} min)')
    print(f'  Train Accuracy : {train_acc:.4f} (Avg Loss: {avg_train_loss:.4f})')
    print(f'  Val Accuracy   : {val_acc:.4f} (Avg Loss: {avg_val_loss:.4f})')
    print('=' * 60 + '
')

    epoch_ckpt = f'text_model_epoch{epoch+1}.pt'
    torch.save(model.state_dict(), epoch_ckpt)
    print(f'Saved: {epoch_ckpt}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'text_model_best.pt')
        print(f'>>> New Best Model saved with Val Acc: {val_acc:.4f}')

    history.append({
        'epoch': epoch + 1,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'time_minutes': epoch_time
    })

# Save final checkpoint
torch.save(model.state_dict(), 'text_model_final.pt')
print('
Final checkpoint saved to text_model_final.pt')
with open('training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Exported training_history.json')

In [ ]:
# CELL 7 — Independent Held-out Test Set Evaluation & Matrix Export
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# Load best checkpoint for evaluation
if os.path.exists('text_model_best.pt'):
    model.load_state_dict(torch.load('text_model_best.pt', map_location=device))
    print('Loaded text_model_best.pt for test evaluation.')
else:
    model.load_state_dict(torch.load('text_model_final.pt', map_location=device))

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(batch['labels'].tolist())

test_acc = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
suicidal_recall = report_dict.get('Suicidal', {}).get('recall', 0.0)

print('
' + '=' * 60)
print('MENTALBERT — FINAL REPRODUCED TEST EVALUATION')
print('=' * 60)
print(f'Test Accuracy        : {test_acc*100:.2f}% ({test_acc:.4f})  [Original Target: 84.26%]')
print(f'Macro F1             : {macro_f1*100:.2f}% ({macro_f1:.4f})  [Original Target: 82.90%]')
print(f'Weighted F1          : {weighted_f1*100:.2f}% ({weighted_f1:.4f})  [Original Target: 84.21%]')
print(f'Suicidal-Class Recall: {suicidal_recall*100:.2f}% ({suicidal_recall:.4f})  [Original Target: 71.00%]')
print('
Full Per-Class Report:')
print(classification_report(all_labels, all_preds, target_names=class_names))

# Export results to JSON
eval_summary = {
    'test_accuracy': test_acc,
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1,
    'suicidal_recall': suicidal_recall,
    'classification_report': report_dict
}
with open('evaluation_summary_text.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)
print('Exported evaluation_summary_text.json')

# Plot and save Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'MentalBERT 7-Class Test Confusion Matrix (Acc: {test_acc*100:.1f}%)')
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.tight_layout()
plt.savefig('confusion_matrix_text.png', dpi=300)
plt.show()
print('Saved confusion_matrix_text.png')

In [ ]:
# CELL 8 — Export Artifacts (Google Drive backup + Browser Download)
import os
import shutil

artifacts = [
    'text_model_final.pt',
    'text_model_best.pt',
    'training_history.json',
    'evaluation_summary_text.json',
    'confusion_matrix_text.png'
]

# Option A: Google Drive backup
try:
    drive_dir = '/content/drive/MyDrive/MINDTRACE_checkpoints/'
    if os.path.exists('/content/drive/MyDrive'):
        os.makedirs(drive_dir, exist_ok=True)
        for art in artifacts:
            if os.path.exists(art):
                shutil.copy(art, drive_dir)
        print(f'Successfully copied all artifacts to Google Drive: {drive_dir}')
except Exception as e:
    print(f'Google Drive note: {e}')

# Option B: Direct Browser Download
try:
    from google.colab import files
    print('Triggering browser download for text_model_final.pt (438 MB)...')
    files.download('text_model_final.pt')
    files.download('evaluation_summary_text.json')
    files.download('confusion_matrix_text.png')
    print('Download triggered! Once downloaded, move text_model_final.pt to:')
    print('  multimodal-mental-health/models/text/text_model_final.pt')
except Exception as e:
    print(f'Manual download note: {e}')
    print('Files available in current working directory: text_model_final.pt')